# LeetCode 698 — Partition to K Equal Sum Subsets bằng Python

Link đề: https://leetcode.com/problems/partition-to-k-equal-sum-subsets/ · Độ khó: **Medium** · Chủ đề: Array, Backtracking (quay lui), Bit Manipulation, Bitmask DP, Memoization

Skeleton LeetCode:

```python
class Solution:
    def canPartitionKSubsets(self, nums: List[int], k: int) -> bool:
        ...
```

Ghi chú: `n <= 16` là dấu hiệu rõ ràng của **bitmask** (mỗi tập con của mảng là một số nguyên `0 .. 2^n - 1`) hoặc **quay lui có cắt tỉa**. Notebook đi từ cách gán mỗi số vào một trong `k` rổ (`k^n`), tới quay lui cắt tỉa, quay lui "dựng từng tập con", bitmask DP và bao phủ chính xác bằng các tập con tốt.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chia mảng `nums` thành đúng `k` tập con **không rỗng** (mỗi phần tử thuộc đúng một tập) sao cho **tổng bằng nhau**.
2. Nắm các công cụ Python cần thiết: `sum`, `sorted(..., reverse=True)`, `%` và `//`, toán tử bit `<<`, `>>`, `&`, `|`, `^`, mẹo `mask & -mask` (bit thấp nhất), `int.bit_count`, và cấu trúc backtracking: chọn -> đệ quy -> **hoàn tác**.
3. Hiểu điều kiện cần: `tổng % k == 0`, mục tiêu mỗi tập `target = tổng / k`, và mọi phần tử `<= target`.
4. Đi qua 5 cách: vét cạn `k^n`, quay lui theo rổ có cắt tỉa (sắp giảm dần + bỏ rổ trùng tổng), quay lui dựng từng tập con bằng mảng `used`, bitmask DP `dp[mask]`, bao phủ chính xác bằng các tập con có tổng `target`; đối chiếu bằng oracle vét cạn.
5. Tránh các lỗi kinh điển: bỏ qua kiểm tra chia hết, quên hoàn tác (backtrack), chọn tham lam "lớn nhất còn vừa", không sắp xếp/không cắt tỉa gây vượt thời gian, nhầm `dp[mask]` là tổng thay vì phần dư.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên `k`. Trả về `true` nếu có thể chia `nums` thành `k` tập con **không rỗng** có **tổng bằng nhau**.

**Ví dụ 1**

- Input: `nums = [4,3,2,3,5,2,1]`, `k = 4`
- Output: `true`
- Giải thích: chia thành `(5)`, `(1, 4)`, `(2, 3)`, `(2, 3)`, mỗi tập tổng 5.

**Ví dụ 2**

- Input: `nums = [1,2,3,4]`, `k = 3`
- Output: `false`

**Ràng buộc:**

- `1 <= k <= nums.length <= 16`
- `1 <= nums[i] <= 10^4`
- Tần suất mỗi phần tử trong khoảng `[1, 4]` (mỗi giá trị xuất hiện 1 tới 4 lần).

Lưu ý: mọi phần tử đều **dương** nên nếu `k` tập có cùng tổng `target > 0` thì mỗi tập chắc chắn không rỗng.

In [1]:
from typing import List, Optional
from functools import lru_cache
from itertools import product
from collections import Counter
import random, time

examples = [
    ([4, 3, 2, 3, 5, 2, 1], 4, True),
    ([1, 2, 3, 4], 3, False),
]
for nums, k, expected in examples:
    total = sum(nums)
    print(f"nums={nums} k={k} tổng={total} tổng%k={total % k} kỳ vọng={expected}")

nums=[4, 3, 2, 3, 5, 2, 1] k=4 tổng=20 tổng%k=0 kỳ vọng=True
nums=[1, 2, 3, 4] k=3 tổng=10 tổng%k=1 kỳ vọng=False


## 3. Kiến thức nền

### 3.1. Điều kiện cần: chia hết và phần tử lớn nhất

Nếu chia được thành `k` tập cùng tổng `target` thì `tổng = k * target`, nên **`tổng % k == 0`** và `target = tổng // k`. Ngoài ra mỗi phần tử nằm trong một tập có tổng `target` nên **`max(nums) <= target`**. Hai kiểm tra rẻ này loại rất nhiều trường hợp.

In [2]:
def necessary(nums, k):
    total = sum(nums)
    if total % k:
        return False, None
    target = total // k
    return max(nums) <= target, target

print(necessary([4, 3, 2, 3, 5, 2, 1], 4))   # (True, 5)
print(necessary([1, 2, 3, 4], 3))            # (False, None): 10 % 3 != 0
print(necessary([1, 1, 8], 2))               # (False, 5): 8 > 5

(True, 5)
(False, None)
(False, 5)


### 3.2. `sorted(reverse=True)` và vì sao xếp giảm dần giúp quay lui

Khi đặt các số vào rổ, ta nên đặt số **lớn trước**: số lớn có ít chỗ hợp lệ nhất, nên sai sớm được phát hiện sớm (cắt cụt cây tìm kiếm).

In [3]:
nums = [4, 3, 2, 3, 5, 2, 1]
print(sorted(nums, reverse=True))
print(sorted(nums), "(tăng dần, ngược với ý định)")

[5, 4, 3, 3, 2, 2, 1]
[1, 2, 2, 3, 3, 4, 5] (tăng dần, ngược với ý định)


### 3.3. Quay lui (backtracking): chọn, đệ quy, hoàn tác

Khuôn mẫu:

```
def dfs(trạng thái):
    if đạt đích: return True
    for lựa chọn hợp lệ:
        áp dụng lựa chọn          # chọn
        if dfs(trạng thái mới): return True
        hoàn tác lựa chọn         # undo: bắt buộc!
    return False
```

Quên **hoàn tác** là lỗi phổ biến nhất: các nhánh sau thấy trạng thái "bẩn" do nhánh trước để lại.

In [4]:
# Ví dụ nhỏ: tìm một tập con của [3, 5, 7, 8] có tổng 12 bằng quay lui
def subset_sum(arr, target):
    chosen = []
    def dfs(i, cur):
        if cur == target:
            return True
        if i == len(arr) or cur > target:
            return False
        chosen.append(arr[i])              # chọn
        if dfs(i + 1, cur + arr[i]):
            return True
        chosen.pop()                       # hoàn tác
        return dfs(i + 1, cur)             # không chọn
    return list(chosen) if dfs(0, 0) else None

print(subset_sum([3, 5, 7, 8], 12), subset_sum([3, 5, 7, 8], 6))

[5, 7] None


### 3.4. Toán tử bit và bitmask

Với `n <= 16`, mỗi **tập con** của `nums` ứng với một số nguyên `mask` trong `[0, 2^n - 1]`: bit thứ `i` bằng 1 nghĩa là phần tử `nums[i]` **được chọn**.

- `1 << i`: số chỉ có bit `i` bằng 1.
- `mask >> i & 1`: kiểm tra bit `i` có bằng 1 (có trong tập).
- `mask | (1 << i)`: thêm phần tử `i`; `mask ^ (1 << i)`: bỏ phần tử `i` (nếu đang có).
- `mask & -mask`: giữ lại **bit thấp nhất** đang bật; `.bit_length() - 1` cho chỉ số của nó.
- `(1 << n) - 1`: tập "tất cả phần tử".
- `mask.bit_count()`: số phần tử trong tập (Python 3.10+).

In [5]:
nums = [4, 3, 2, 3, 5, 2, 1]
n = len(nums)
mask = (1 << 0) | (1 << 4) | (1 << 6)           # chọn nums[0], nums[4], nums[6]
print(bin(mask), [nums[i] for i in range(n) if mask >> i & 1], "tổng =", sum(nums[i] for i in range(n) if mask >> i & 1))
print("bit thấp nhất:", bin(mask & -mask), "-> chỉ số", (mask & -mask).bit_length() - 1)
print("bỏ phần tử 4:", bin(mask ^ (1 << 4)), "| số phần tử:", mask.bit_count())
print("tập đầy đủ:", bin((1 << n) - 1))

0b1010001 [4, 5, 1] tổng = 10
bit thấp nhất: 0b1 -> chỉ số 0
bỏ phần tử 4: 0b1000001 | số phần tử: 3
tập đầy đủ: 0b1111111


### 3.5. Tổng của mọi tập con bằng bit thấp nhất

`s[mask] = s[mask ^ low] + nums[chỉ số của low]` với `low = mask & -mask`: tổng của `mask` bằng tổng của `mask` bỏ bit thấp nhất, cộng phần tử đó. Tính cả `2^n` tổng trong `O(2^n)`.

In [6]:
nums = [1, 2, 3]
n = len(nums)
s = [0] * (1 << n)
for mask in range(1, 1 << n):
    low = mask & -mask
    s[mask] = s[mask ^ low] + nums[low.bit_length() - 1]
for mask in range(1 << n):
    print(f"{mask:0{n}b} -> {[nums[i] for i in range(n) if mask >> i & 1]} tổng {s[mask]}")

000 -> [] tổng 0
001 -> [1] tổng 1
010 -> [2] tổng 2
011 -> [1, 2] tổng 3
100 -> [3] tổng 3
101 -> [1, 3] tổng 4
110 -> [2, 3] tổng 5
111 -> [1, 2, 3] tổng 6


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `nums = [4,3,2,3,5,2,1]`, `k = 4`. Tổng = 20, `20 % 4 = 0`, nên `target = 5`. Max = 5 <= 5, qua cả hai điều kiện. Dựng thủ công: số `5` đã đủ 5 một mình; `4` cần thêm `1`; còn `3, 3, 2, 2`: `3 + 2` và `3 + 2`. Tập: `(5)`, `(4,1)`, `(3,2)`, `(3,2)`.

Ví dụ 2: `[1,2,3,4]`, `k = 3`: tổng 10, `10 % 3 != 0` nên **ngay lập tức** `false`, không cần thử.

Vậy bài là: với `target` cố định, có thể **phủ** mọi phần tử bằng `k` nhóm có tổng đúng `target` không? Đây là bài toán NP-khó tổng quát (giống "bin packing" đúng đầy), nhưng `n <= 16` đủ nhỏ.

In [7]:
nums, k = [4, 3, 2, 3, 5, 2, 1], 4
ok, target = necessary(nums, k)
print("đủ điều kiện cần?", ok, "target =", target)
groups = [[5], [4, 1], [3, 2], [3, 2]]
print("nhóm thủ công:", groups, "tổng từng nhóm:", [sum(g) for g in groups])
assert sorted(sum(groups, [])) == sorted(nums) and all(sum(g) == target for g in groups)

đủ điều kiện cần? True target = 5
nhóm thủ công: [[5], [4, 1], [3, 2], [3, 2]] tổng từng nhóm: [5, 5, 5, 5]


## 5. Cách 1 — Brute Force: gán mỗi số vào một trong `k` rổ

**Ý tưởng.** Mỗi phần tử có `k` lựa chọn rổ, tổng cộng `k^n` cách gán. Với mỗi cách, kiểm tra `k` rổ có tổng bằng nhau không. Đây là định nghĩa trực tiếp của bài toán và cũng là **oracle** (chắc chắn đúng, chỉ dùng cho `n` nhỏ).

In [8]:
def can_brute(nums: List[int], k: int) -> bool:
    n = len(nums)
    for assign in product(range(k), repeat=n):          # assign[i] = rổ của nums[i]
        sums = [0] * k
        for v, b in zip(nums, assign):
            sums[b] += v
        if len(set(sums)) == 1:                         # mọi rổ cùng tổng (rổ rỗng có tổng 0 khác tổng dương)
            return True
    return False

for nums, k, expected in examples:
    got = can_brute(nums, k) if len(nums) <= 7 else None
    print(nums, k, "->", got)
    if got is not None:
        assert got == expected
print("k^n với ví dụ 1 là", 4 ** 7, "cách gán")

[4, 3, 2, 3, 5, 2, 1] 4 -> True
[1, 2, 3, 4] 3 -> False
k^n với ví dụ 1 là 16384 cách gán


**Độ phức tạp và nút thắt.** `O(k^n * n)` thời gian. Với `n = 16, k = 4` là `4^16 ≈ 4,3 tỷ`: không thể. Nút thắt: ta thử **cả những cách gán vô nghĩa** (ví dụ rổ nào đã vượt `target` rồi mà vẫn tiếp tục) và **các cách gán đối xứng** (hoán đổi nhãn hai rổ giống hệt nhau cho kết quả giống hệt).

In [9]:
print(f"{'n':>3} {'k':>3} {'k^n':>14}")
for n, k in [(7, 4), (10, 3), (12, 4), (16, 4), (16, 8)]:
    print(f"{n:>3} {k:>3} {k ** n:>14,}")

  n   k            k^n
  7   4         16,384
 10   3         59,049
 12   4     16,777,216
 16   4  4,294,967,296
 16   8 281,474,976,710,656


## 6. Ý tưởng thuật toán / bước đột phá

Ba ý tưởng cắt giảm khối lượng, mỗi ý ứng với một cách:

1. **Điều kiện cần + cắt tỉa:** kiểm tra `tổng % k` và `max <= target`; khi đặt số vào rổ, **bỏ nhánh làm rổ vượt `target`**; **sắp giảm dần**; **bỏ rổ có cùng tổng với rổ trước** (đối xứng) -> Cách 2.
2. **Đổi cách nhìn:** thay vì "phần tử chọn rổ", hãy **dựng từng tập con** có tổng `target`: chọn các phần tử theo thứ tự chỉ số tăng (tổ hợp, không phải hoán vị), khi đủ `target` thì sang rổ mới -> Cách 3.
3. **Trạng thái là tập đã dùng:** với `n <= 16` mỗi tập đã dùng là một `mask`, và **phần dư của rổ hiện tại** là `tổng(mask) % target` (xác định bởi `mask`). Do đó chỉ có `2^n` trạng thái: bitmask DP -> Cách 4; hoặc đếm trực tiếp các tập con tốt (tổng đúng `target`) và phủ chính xác -> Cách 5.

| Cách | Ý tưởng |
|---|---|
| 2 | Quay lui: số -> rổ, cắt tỉa (xếp giảm dần, bỏ rổ trùng tổng) |
| 3 | Quay lui: dựng từng tập con bằng mảng `used`, rổ này đầy mới sang rổ khác |
| 4 | Bitmask DP bottom-up: `dp[mask]` = phần dư rổ hiện tại, `-1` nếu không đạt được |
| 5 | Bao phủ chính xác: các tập con có tổng `target`, luôn chọn tập chứa bit thấp nhất còn lại |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Quay lui theo rổ, có cắt tỉa

**Giải thích từng đoạn.**

- Kiểm tra `tổng % k` và `max <= target`.
- `cur[j]` là tổng hiện tại của rổ `j`. Sắp `nums` giảm dần; `dfs(i)` đặt `nums[i]` vào lần lượt các rổ.
- **Cắt tỉa 1:** nếu `cur[j] + nums[i] > target` thì bỏ.
- **Cắt tỉa 2 (đối xứng):** nếu `cur[j] == cur[j - 1]` thì rổ `j` giống rổ `j - 1` về mặt trạng thái; thử rổ `j-1` đã thất bại thì thử `j` cũng thất bại -> bỏ.
- Sau khi thử xong một rổ phải **hoàn tác** `cur[j] -= nums[i]`.
- Đặt hết `n` số mà không vi phạm: vì tổng bằng `k * target` và mỗi rổ `<= target` nên mọi rổ đều đúng `target`.

In [10]:
def can_buckets(nums: List[int], k: int) -> bool:
    total = sum(nums)
    if total % k:
        return False
    target = total // k
    nums = sorted(nums, reverse=True)
    if nums[0] > target:
        return False
    cur = [0] * k

    def dfs(i: int) -> bool:
        if i == len(nums):
            return True
        for j in range(k):
            if j and cur[j] == cur[j - 1]:        # rổ trùng trạng thái với rổ trước: bỏ
                continue
            if cur[j] + nums[i] > target:          # vượt mục tiêu: bỏ
                continue
            cur[j] += nums[i]
            if dfs(i + 1):
                return True
            cur[j] -= nums[i]                      # hoàn tác
        return False

    return dfs(0)

for nums, k, expected in examples:
    got = can_buckets(nums, k)
    print(nums, k, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[4, 3, 2, 3, 5, 2, 1] 4 -> True OK
[1, 2, 3, 4] 3 -> False OK


**Code nộp LeetCode (Cách 2):**

In [11]:
class Solution:
    def canPartitionKSubsets(self, nums: List[int], k: int) -> bool:
        total = sum(nums)
        if total % k:
            return False
        s = total // k
        nums = sorted(nums, reverse=True)
        if nums[0] > s:
            return False
        cur = [0] * k

        def dfs(i: int) -> bool:
            if i == len(nums):
                return True
            for j in range(k):
                if j and cur[j] == cur[j - 1]:
                    continue
                if cur[j] + nums[i] > s:
                    continue
                cur[j] += nums[i]
                if dfs(i + 1):
                    return True
                cur[j] -= nums[i]
            return False

        return dfs(0)

sol_buckets = Solution().canPartitionKSubsets
print([sol_buckets(nums, k) for nums, k, _ in examples])

[True, False]


**Độ phức tạp (Cách 2):** xấu nhất vẫn `O(k^n)` nhưng cắt tỉa làm thực tế nhỏ hơn nhiều; bộ nhớ `O(n + k)` (ngăn xếp đệ quy + `cur`).

---

### 7.2. Cách 3 — Quay lui dựng từng tập con (mảng `used`)

**Ý tưởng khác.** Dựng **một tập có tổng `target` tại một thời điểm**. `dfs(start, cur, left)`: `left` là số rổ còn phải dựng, `cur` là tổng rổ đang dựng, `start` là chỉ số bắt đầu xét (chỉ xét chỉ số tăng dần để tránh thử cùng một tập con theo nhiều thứ tự). Khi `cur == target` thì sang rổ mới (`start = 0`, `cur = 0`, `left - 1`).

**Giải thích từng đoạn.**

- Khi `left == 1`: các phần tử còn lại có tổng đúng `target` (vì `tổng = k * target` và `k - 1` rổ đã đủ) nên thành công.
- Duyệt `i` từ `start`: bỏ qua phần tử đã dùng, hoặc làm `cur` vượt `target`.
- **Bỏ trùng giá trị:** nếu `nums[i] == nums[i-1]` và `nums[i-1]` chưa dùng và `i > start` thì thử `nums[i]` thừa (vì `nums[i-1]` giống hệt đã thử và thất bại).
- Đánh dấu `used[i]`, đệ quy với `start = i + 1`, rồi **hoàn tác**.
- Nếu `cur == 0` (phần tử đầu của rổ, tức phần tử lớn nhất còn lại) mà thất bại thì cả nhánh thất bại: phần tử lớn nhất còn lại buộc phải thuộc **một** rổ nào đó, và rổ nào cũng như nhau.

In [12]:
def can_subsets(nums: List[int], k: int) -> bool:
    total = sum(nums)
    if total % k:
        return False
    target = total // k
    nums = sorted(nums, reverse=True)
    if nums[0] > target:
        return False
    n = len(nums)
    used = [False] * n

    def dfs(start: int, cur: int, left: int) -> bool:
        if left == 1:
            return True
        if cur == target:
            return dfs(0, 0, left - 1)
        for i in range(start, n):
            if used[i] or cur + nums[i] > target:
                continue
            if i > start and nums[i] == nums[i - 1] and not used[i - 1]:
                continue
            used[i] = True
            if dfs(i + 1, cur + nums[i], left):
                return True
            used[i] = False
            if cur == 0:                      # phần tử đầu của rổ thất bại -> cả nhánh thất bại
                return False
        return False

    return dfs(0, 0, k)

for nums, k, expected in examples:
    assert can_subsets(nums, k) == expected
print("dựng từng tập con OK")

dựng từng tập con OK


**Code nộp LeetCode (Cách 3):**

In [13]:
class Solution:
    def canPartitionKSubsets(self, nums: List[int], k: int) -> bool:
        total = sum(nums)
        if total % k:
            return False
        target = total // k
        nums = sorted(nums, reverse=True)
        if nums[0] > target:
            return False
        n = len(nums)
        used = [False] * n

        def dfs(start: int, cur: int, left: int) -> bool:
            if left == 1:
                return True
            if cur == target:
                return dfs(0, 0, left - 1)
            for i in range(start, n):
                if used[i] or cur + nums[i] > target:
                    continue
                if i > start and nums[i] == nums[i - 1] and not used[i - 1]:
                    continue
                used[i] = True
                if dfs(i + 1, cur + nums[i], left):
                    return True
                used[i] = False
                if cur == 0:
                    return False
            return False

        return dfs(0, 0, k)

sol_subsets = Solution().canPartitionKSubsets
print([sol_subsets(nums, k) for nums, k, _ in examples])

[True, False]


**Độ phức tạp (Cách 3):** xấu nhất `O(k * 2^n)`-cỡ thực tế (mỗi tập con được duyệt theo thứ tự chỉ số tăng, nhưng vẫn có thể mũ); bộ nhớ `O(n)`.

---

### 7.3. Cách 4 — Bitmask DP (bottom-up)

**Ý tưởng.** Gọi `dp[mask]` = **phần dư của rổ đang dựng** (tổng các phần tử trong `mask` theo `% target`) nếu có **một thứ tự xếp** các phần tử của `mask` sao cho không bao giờ vượt `target` khi cộng dần; ngược lại `-1`.

**Giải thích từng đoạn.**

- `dp[0] = 0`.
- Với mỗi `mask` đạt được (`dp[mask] >= 0`) và mỗi phần tử `i` chưa có trong `mask`: nếu `dp[mask] + nums[i] <= target` thì `mask | (1 << i)` đạt được với `dp = (dp[mask] + nums[i]) % target`.
- Khi `dp[mask] + nums[i] == target`, ta đóng một rổ và phần dư về 0.
- Đáp án: `dp[full] == 0` (đạt được và vừa hết các rổ).
- Vì phần dư xác định bởi **tổng** của `mask` (`s[mask] % target`), gán `dp[nm]` lần đầu là đủ.

In [14]:
def can_bitmask(nums: List[int], k: int) -> bool:
    total = sum(nums)
    if total % k:
        return False
    target = total // k
    if max(nums) > target:
        return False
    n = len(nums)
    full = (1 << n) - 1
    dp = [-1] * (1 << n)
    dp[0] = 0
    for mask in range(1 << n):
        if dp[mask] < 0:
            continue
        for i in range(n):
            if mask >> i & 1:
                continue
            nm = mask | (1 << i)
            if dp[nm] < 0 and dp[mask] + nums[i] <= target:
                dp[nm] = (dp[mask] + nums[i]) % target
    return dp[full] == 0

for nums, k, expected in examples:
    assert can_bitmask(nums, k) == expected
print("bitmask DP OK")

bitmask DP OK


**Code nộp LeetCode (Cách 4):**

In [15]:
class Solution:
    def canPartitionKSubsets(self, nums: List[int], k: int) -> bool:
        total = sum(nums)
        if total % k:
            return False
        target = total // k
        if max(nums) > target:
            return False
        n = len(nums)
        dp = [-1] * (1 << n)
        dp[0] = 0
        for mask in range(1 << n):
            if dp[mask] < 0:
                continue
            for i in range(n):
                if mask >> i & 1:
                    continue
                nm = mask | (1 << i)
                if dp[nm] < 0 and dp[mask] + nums[i] <= target:
                    dp[nm] = (dp[mask] + nums[i]) % target
        return dp[(1 << n) - 1] == 0

sol_bitmask = Solution().canPartitionKSubsets
print([sol_bitmask(nums, k) for nums, k, _ in examples])

[True, False]


**Độ phức tạp (Cách 4):** thời gian `O(n * 2^n)` (với `n = 16`: khoảng một triệu phép), bộ nhớ `O(2^n)`.

---

### 7.4. Cách 5 — Bao phủ chính xác bằng các tập con "tốt"

**Ý tưởng khác.** Gọi một tập con là **tốt** nếu tổng của nó bằng đúng `target`. Cần phủ **mọi phần tử** bằng các tập tốt **rời nhau** (và tự nhiên đúng `k` tập vì `tổng = k * target`). Luôn chọn tập tốt **chứa phần tử có bit thấp nhất** trong số còn lại (để tránh đếm lặp thứ tự).

**Giải thích từng đoạn.**

- Tính `s[mask]` cho mọi `mask` (mục 3.5), lọc `good` = các `mask` có `s[mask] == target`, nhóm theo chỉ số bit thấp nhất của chúng.
- `dfs(rem)`: `rem` là tập phần tử chưa phủ. `rem == 0` -> thành công. Lấy `low = rem & -rem`; thử mọi tập tốt `g` có cùng bit thấp nhất (`g & low`) và `g & rem == g` (nằm trọn trong `rem`); nếu `dfs(rem ^ g)` thành công thì thành công.
- Ghi nhớ các `rem` đã thất bại (`failed`) để không thử lại.

In [16]:
def can_cover(nums: List[int], k: int) -> bool:
    total = sum(nums)
    if total % k:
        return False
    target = total // k
    if max(nums) > target:
        return False
    n = len(nums)
    s = [0] * (1 << n)
    by_low = {}
    for mask in range(1, 1 << n):
        low = mask & -mask
        s[mask] = s[mask ^ low] + nums[low.bit_length() - 1]
        if s[mask] == target:
            by_low.setdefault(low, []).append(mask)
    failed = set()

    def dfs(rem: int) -> bool:
        if rem == 0:
            return True
        if rem in failed:
            return False
        low = rem & -rem
        for g in by_low.get(low, []):
            if g & rem == g and dfs(rem ^ g):
                return True
        failed.add(rem)
        return False

    return dfs((1 << n) - 1)

for nums, k, expected in examples:
    assert can_cover(nums, k) == expected
print("bao phủ chính xác OK")

bao phủ chính xác OK


**Code nộp LeetCode (Cách 5):**

In [17]:
class Solution:
    def canPartitionKSubsets(self, nums: List[int], k: int) -> bool:
        total = sum(nums)
        if total % k:
            return False
        target = total // k
        if max(nums) > target:
            return False
        n = len(nums)
        s = [0] * (1 << n)
        by_low = {}
        for mask in range(1, 1 << n):
            low = mask & -mask
            s[mask] = s[mask ^ low] + nums[low.bit_length() - 1]
            if s[mask] == target:
                by_low.setdefault(low, []).append(mask)
        failed = set()

        def dfs(rem: int) -> bool:
            if rem == 0:
                return True
            if rem in failed:
                return False
            low = rem & -rem
            for g in by_low.get(low, []):
                if g & rem == g and dfs(rem ^ g):
                    return True
            failed.add(rem)
            return False

        return dfs((1 << n) - 1)

sol_cover = Solution().canPartitionKSubsets
print([sol_cover(nums, k) for nums, k, _ in examples])

[True, False]


**Độ phức tạp (Cách 5):** tiền xử lý `O(2^n)`; tìm kiếm tối đa `2^n` trạng thái `rem`, mỗi trạng thái duyệt các tập tốt có bit thấp nhất tương ứng (thường rất ít). Bộ nhớ `O(2^n)`.

## 8. TRACE từng bước

### 8.1. Cách 2 trên ví dụ 1

Sau khi sắp giảm dần: `[5, 4, 3, 3, 2, 2, 1]`, `target = 5`, `k = 4`. Hàm `trace_buckets` in từng lần đặt (`+`), bỏ qua (`skip`) và hoàn tác (`-`), thụt lề theo độ sâu.

In [18]:
def trace_buckets(nums, k, limit=60):
    total = sum(nums)
    target = total // k
    nums = sorted(nums, reverse=True)
    cur = [0] * k
    print(f"nums (giảm dần) = {nums}, target = {target}")
    lines = [0]

    def log(msg):
        if lines[0] < limit:
            print(msg)
        lines[0] += 1

    def dfs(i):
        if i == len(nums):
            log(f"{'  ' * i}-> đặt xong mọi số: cur = {cur}")
            return True
        for j in range(k):
            pad = "  " * i
            if j and cur[j] == cur[j - 1]:
                log(f"{pad}số {nums[i]} -> rổ {j}: skip (cùng tổng với rổ {j - 1})")
                continue
            if cur[j] + nums[i] > target:
                log(f"{pad}số {nums[i]} -> rổ {j}: skip (vượt {target})")
                continue
            cur[j] += nums[i]
            log(f"{pad}số {nums[i]} -> rổ {j}: +  cur = {cur}")
            if dfs(i + 1):
                return True
            cur[j] -= nums[i]
            log(f"{pad}số {nums[i]} <- rổ {j}: -  (hoàn tác) cur = {cur}")
        return False

    res = dfs(0)
    if lines[0] > limit:
        print(f"... (còn {lines[0] - limit} dòng nữa)")
    return res

assert trace_buckets([4, 3, 2, 3, 5, 2, 1], 4) is True

nums (giảm dần) = [5, 4, 3, 3, 2, 2, 1], target = 5
số 5 -> rổ 0: +  cur = [5, 0, 0, 0]
  số 4 -> rổ 0: skip (vượt 5)
  số 4 -> rổ 1: +  cur = [5, 4, 0, 0]
    số 3 -> rổ 0: skip (vượt 5)
    số 3 -> rổ 1: skip (vượt 5)
    số 3 -> rổ 2: +  cur = [5, 4, 3, 0]
      số 3 -> rổ 0: skip (vượt 5)
      số 3 -> rổ 1: skip (vượt 5)
      số 3 -> rổ 2: skip (vượt 5)
      số 3 -> rổ 3: +  cur = [5, 4, 3, 3]
        số 2 -> rổ 0: skip (vượt 5)
        số 2 -> rổ 1: skip (vượt 5)
        số 2 -> rổ 2: +  cur = [5, 4, 5, 3]
          số 2 -> rổ 0: skip (vượt 5)
          số 2 -> rổ 1: skip (vượt 5)
          số 2 -> rổ 2: skip (vượt 5)
          số 2 -> rổ 3: +  cur = [5, 4, 5, 5]
            số 1 -> rổ 0: skip (vượt 5)
            số 1 -> rổ 1: +  cur = [5, 5, 5, 5]
              -> đặt xong mọi số: cur = [5, 5, 5, 5]


### 8.2. Cách 4 (bitmask DP) trên `nums = [3, 1, 2, 2]`, `k = 2`

Tổng 8, `target = 4`. Liệt kê `dp[mask]` cho mọi `mask` (bit `i` ứng với `nums[i]`). `-1` nghĩa là không có thứ tự xếp hợp lệ; số khác là phần dư của rổ đang dựng. Đáp án: `dp[1111] == 0`.

In [19]:
def trace_bitmask(nums, k):
    total = sum(nums)
    target = total // k
    n = len(nums)
    dp = [-1] * (1 << n)
    dp[0] = 0
    for mask in range(1 << n):
        if dp[mask] < 0:
            continue
        for i in range(n):
            if mask >> i & 1:
                continue
            nm = mask | (1 << i)
            if dp[nm] < 0 and dp[mask] + nums[i] <= target:
                dp[nm] = (dp[mask] + nums[i]) % target
    print(f"nums = {nums}, target = {target}")
    for mask in range(1 << n):
        members = [nums[i] for i in range(n) if mask >> i & 1]
        print(f"mask {mask:0{n}b}  phần tử {str(members):14} tổng {sum(members):2}  dp = {dp[mask]:2}")
    return dp[(1 << n) - 1] == 0

assert trace_bitmask([3, 1, 2, 2], 2) is True

nums = [3, 1, 2, 2], target = 4
mask 0000  phần tử []             tổng  0  dp =  0
mask 0001  phần tử [3]            tổng  3  dp =  3
mask 0010  phần tử [1]            tổng  1  dp =  1
mask 0011  phần tử [3, 1]         tổng  4  dp =  0
mask 0100  phần tử [2]            tổng  2  dp =  2
mask 0101  phần tử [3, 2]         tổng  5  dp = -1
mask 0110  phần tử [1, 2]         tổng  3  dp =  3
mask 0111  phần tử [3, 1, 2]      tổng  6  dp =  2
mask 1000  phần tử [2]            tổng  2  dp =  2
mask 1001  phần tử [3, 2]         tổng  5  dp = -1
mask 1010  phần tử [1, 2]         tổng  3  dp =  3
mask 1011  phần tử [3, 1, 2]      tổng  6  dp =  2
mask 1100  phần tử [2, 2]         tổng  4  dp =  0
mask 1101  phần tử [3, 2, 2]      tổng  7  dp =  3
mask 1110  phần tử [1, 2, 2]      tổng  5  dp =  1
mask 1111  phần tử [3, 1, 2, 2]   tổng  8  dp =  0


Một số dòng đáng chú ý: `mask 0101` (phần tử `3` và `2`) có tổng 5 > 4 nên `dp = -1` (không thể nằm chung một rổ); `mask 0011` (`3` và `1`) tổng 4 -> `dp = 0` (rổ vừa đủ, đã đóng). Hãy so sánh `dp[1111] = 0`.

### 8.3. Cách 5: các tập tốt và bao phủ

Với `nums = [3, 1, 2, 2]`, `target = 4`, các tập tốt là `{3, 1}` (`mask 0011`) và `{2, 2}` (`mask 1100`). Phủ: bit thấp nhất của `1111` là bit 0, thử tập tốt chứa bit 0: `0011`, còn lại `1100` là tập tốt -> xong.

In [20]:
def trace_cover(nums, k):
    total = sum(nums)
    target = total // k
    n = len(nums)
    s = [0] * (1 << n)
    good = []
    for mask in range(1, 1 << n):
        low = mask & -mask
        s[mask] = s[mask ^ low] + nums[low.bit_length() - 1]
        if s[mask] == target:
            good.append(mask)
    print("tập tốt:", [f"{g:0{n}b}" for g in good])
    def dfs(rem, depth=0):
        pad = "  " * depth
        if rem == 0:
            print(f"{pad}rem = 0 -> phủ xong")
            return True
        low = rem & -rem
        print(f"{pad}rem = {rem:0{n}b}, bit thấp nhất = {low:0{n}b}")
        for g in good:
            if g & low and g & rem == g:
                print(f"{pad}  thử tập tốt {g:0{n}b}")
                if dfs(rem ^ g, depth + 1):
                    return True
        print(f"{pad}  không tập nào phủ được")
        return False
    return dfs((1 << n) - 1)

assert trace_cover([3, 1, 2, 2], 2) is True
print()
assert trace_cover([3, 3, 3, 1], 2) is False

tập tốt: ['0011', '1100']
rem = 1111, bit thấp nhất = 0001
  thử tập tốt 0011
  rem = 1100, bit thấp nhất = 0100
    thử tập tốt 1100
    rem = 0 -> phủ xong

tập tốt: []
rem = 1111, bit thấp nhất = 0001
  không tập nào phủ được


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (quay lui theo rổ, có cắt tỉa): ngắn, là lời giải chuẩn và đủ nhanh với `n <= 16`. Cách 4 (bitmask DP) là phương án dự phòng có thời gian **ổn định** `O(n 2^n)`, không phụ thuộc may rủi của cắt tỉa.

```python
class Solution:
    def canPartitionKSubsets(self, nums: List[int], k: int) -> bool:
        total = sum(nums)
        if total % k:
            return False
        s = total // k
        nums = sorted(nums, reverse=True)
        if nums[0] > s:
            return False
        cur = [0] * k

        def dfs(i: int) -> bool:
            if i == len(nums):
                return True
            for j in range(k):
                if j and cur[j] == cur[j - 1]:
                    continue
                if cur[j] + nums[i] > s:
                    continue
                cur[j] += nums[i]
                if dfs(i + 1):
                    return True
                cur[j] -= nums[i]
            return False

        return dfs(0)
```

In [21]:
class Solution:
    def canPartitionKSubsets(self, nums: List[int], k: int) -> bool:
        total = sum(nums)
        if total % k:
            return False
        s = total // k
        nums = sorted(nums, reverse=True)
        if nums[0] > s:
            return False
        cur = [0] * k

        def dfs(i: int) -> bool:
            if i == len(nums):
                return True
            for j in range(k):
                if j and cur[j] == cur[j - 1]:
                    continue
                if cur[j] + nums[i] > s:
                    continue
                cur[j] += nums[i]
                if dfs(i + 1):
                    return True
                cur[j] -= nums[i]
            return False

        return dfs(0)

sol = Solution()
for nums, k, expected in examples:
    got = sol.canPartitionKSubsets(nums, k)
    print(nums, k, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[4, 3, 2, 3, 5, 2, 1] 4 -> True OK
[1, 2, 3, 4] 3 -> False OK


## 10. Vì sao đúng

**(a) Điều kiện cần.** Nếu chia được thành `k` tập cùng tổng `T` thì tổng toàn mảng `= kT`, nên `tổng % k == 0` và `T = tổng / k`. Mỗi phần tử thuộc một tập có tổng `T` và mọi phần tử dương nên mỗi phần tử `<= T`.

**(b) Cách 2.** Khi không có cắt tỉa, `dfs` thử **mọi** cách gán (`k^n`); cuối cùng nếu không rổ nào vượt `T` và tổng là `kT` thì **mọi rổ đúng `T`** (nếu một rổ `< T` thì rổ khác `> T`, mâu thuẫn). Hai cắt tỉa không mất nghiệm: (i) rổ vượt `T` không bao giờ quay lại hợp lệ vì phần tử dương; (ii) hai rổ có cùng tổng hiện tại là **đối xứng** (hoán đổi chúng cho cùng tập trạng thái tương lai), nên thử một rổ là đủ. Sắp giảm dần chỉ đổi thứ tự thử, không đổi tập nghiệm.

**(c) Cách 3.** Một phân hoạch hợp lệ có thể được liệt kê theo thứ tự: rổ chứa phần tử lớn nhất còn lại trước. Dựng rổ bằng chỉ số tăng liệt kê mỗi tập con đúng một lần. Khi `left == 1`, tổng phần còn lại `= T` (vì `k - 1` rổ đã dùng `(k-1)T`), nên rổ cuối đương nhiên đúng. Cắt tỉa `cur == 0` đúng vì phần tử đầu (lớn nhất còn lại) phải thuộc một rổ, và các rổ chưa dựng là như nhau nên thử rổ này thất bại thì mọi rổ đều thất bại. Cắt tỉa giá trị trùng đúng vì hoán đổi hai phần tử cùng giá trị không đổi tập nghiệm.

**(d) Cách 4.** Với `mask` bất kỳ, tổng `s[mask]` cố định, nên nếu mọi rổ đã đóng thì rổ hiện tại có tổng `s[mask] mod T` **bất kể cách xếp**; điều duy nhất phụ thuộc cách xếp là "có xếp được mà không vượt `T` ở bước nào không". `dp[mask] >= 0` đúng bằng điều đó, và quy nạp theo số bit cho thấy gán lần đầu cho `dp[nm]` luôn đưa ra cùng giá trị `s[nm] mod T`. Đáp án `dp[full] == 0` nghĩa là xếp hết mà phần dư cuối cùng bằng 0, tức mọi rổ đúng `T`.

**(e) Cách 5.** Một phân hoạch hợp lệ là một họ `k` tập tốt rời nhau phủ hết. Luôn xét tập chứa bit thấp nhất còn lại là liệt kê mỗi phân hoạch đúng một lần (không phụ thuộc thứ tự các tập).

**Kiểm chứng bằng máy:** so mọi cách với oracle vét cạn `k^n` trên **mọi** mảng nhỏ (độ dài tới 6, giá trị nhỏ, mọi `k`), rồi so các cách nhanh trên nhiều mảng độ dài tới 16.

In [22]:
solutions = {
    "brute": can_brute,
    "buckets": can_buckets,
    "subsets": can_subsets,
    "bitmask": can_bitmask,
    "cover": can_cover,
    "sol_buckets": sol_buckets,
    "sol_subsets": sol_subsets,
    "sol_bitmask": sol_bitmask,
    "sol_cover": sol_cover,
    "final": Solution().canPartitionKSubsets,
}

def all_arrays(n, vmax):
    return product(range(1, vmax + 1), repeat=n)

checked = 0
n_true = 0
for n, vmax in [(1, 4), (2, 4), (3, 4), (4, 4), (5, 3), (6, 2)]:
    for arr in all_arrays(n, vmax):
        arr = list(arr)
        for k in range(1, n + 1):
            ref = can_brute(arr, k)
            for name, fn in solutions.items():
                assert fn(arr, k) == ref, (name, arr, k)
            checked += 1
            n_true += ref
print(f"mọi mảng độ dài 1..4 giá trị 1..4, độ dài 5 giá trị 1..3, độ dài 6 giá trị 1..2, mọi k: {checked} ca, {n_true} đúng/{checked - n_true} sai; {len(solutions)} cài đặt khớp oracle")

mọi mảng độ dài 1..4 giá trị 1..4, độ dài 5 giá trị 1..3, độ dài 6 giá trị 1..2, mọi k: 2851 ca, 1096 đúng/1755 sai; 10 cài đặt khớp oracle


## 11. Time / Space Complexity

Gọi `n = len(nums)` (`<= 16`) và `k` là số tập.

| Cách | Time | Space |
|---|---|---|
| 1. Gán mọi số vào rổ | `O(k^n * n)` | `O(n)` |
| 2. Quay lui + cắt tỉa | xấu nhất `O(k^n)`, thực tế nhỏ hơn nhiều | `O(n + k)` |
| 3. Dựng từng tập con | xấu nhất mũ, thực tế nhỏ | `O(n)` |
| 4. Bitmask DP | `O(n * 2^n)` | `O(2^n)` |
| 5. Bao phủ chính xác | `O(2^n)` tiền xử lý + `O(2^n * số tập tốt)` | `O(2^n)` |

Với `n = 16`, `2^n * n ≈ 1 triệu`. Đo thử trên các mảng 16 phần tử "khó" (đúng ràng buộc tần suất `<= 4`):

In [23]:
def make_true_instance(rng, n, k, target):
    """Sinh mảng n phần tử chia được thành k tập cùng tổng target (tần suất mỗi giá trị <= 4)."""
    for _ in range(300):
        sizes = [1] * k
        for _ in range(n - k):
            sizes[rng.randrange(k)] += 1
        arr = []
        ok = True
        for m in sizes:
            if m > target:
                ok = False
                break
            cuts = sorted(rng.sample(range(1, target), m - 1)) if m > 1 else []
            parts = [b - a for a, b in zip([0] + cuts, cuts + [target])]
            arr += parts
        if ok and max(Counter(arr).values()) <= 4:
            rng.shuffle(arr)
            return arr
    raise ValueError("không sinh được mảng thỏa tần suất")

rng = random.Random(698)
hard = []
for k, target in [(4, 40), (5, 30), (8, 25), (6, 50), (3, 100), (2, 300)]:
    for _ in range(4):
        hard.append((make_true_instance(rng, 16, k, target), k))
# thêm các biến thể sai: tăng một phần tử lên 1 (tổng đổi nên có thể chia hết hoặc không)
hard += [(h[:-1] + [h[-1] + 1], k) for h, k in hard[:8]]

for name in ["buckets", "subsets", "bitmask", "cover"]:
    t0 = time.perf_counter()
    res = [solutions[name](arr, k) for arr, k in hard]
    dt = (time.perf_counter() - t0) * 1000
    print(f"{name:8} {len(hard)} mảng n=16: {dt:9.1f} ms, số True = {sum(res)}")
ref = [solutions["bitmask"](arr, k) for arr, k in hard]
for name in ["buckets", "subsets", "cover"]:
    assert [solutions[name](arr, k) for arr, k in hard] == ref, name
print("khớp nhau trên các mảng 16 phần tử")

buckets  32 mảng n=16:       0.7 ms, số True = 24
subsets  32 mảng n=16:       0.2 ms, số True = 24


bitmask  32 mảng n=16:    1280.5 ms, số True = 24
cover    32 mảng n=16:     174.6 ms, số True = 24


khớp nhau trên các mảng 16 phần tử


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai, đối chiếu mọi cách với oracle vét cạn trên mảng nhỏ, so khớp các cách nhanh trên mảng tới 16 phần tử, và test ngẫu nhiên có seed (đúng ràng buộc).

In [24]:
# 12.1 ví dụ đề
for nums, k, expected in examples:
    for name, fn in solutions.items():
        if name == "brute" and len(nums) > 7:
            continue
        assert fn(nums, k) == expected, (name, nums, k)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([1], 1, True),                                   # một phần tử, một tập
    ([5, 5], 2, True),                                # k = n, mọi phần tử bằng nhau
    ([5, 6], 2, False),                               # k = n, khác nhau
    ([1, 2, 3, 4], 1, True),                          # k = 1 luôn đúng
    ([1, 2, 3, 4], 2, True),                          # (1,4) (2,3)
    ([1, 2, 3, 4], 4, False),                         # k = n nhưng khác nhau
    ([1, 1, 1, 1, 1], 2, False),                      # tổng 5 không chia hết 2
    ([2, 2, 2, 2, 3, 3], 3, False),                   # tổng 14 không chia hết 3
    ([1, 1, 8], 2, False),                            # target 5 < 8
    ([4, 4, 6, 2, 2, 2], 3, False),                   # tổng 20 không chia hết 3
    ([4, 4, 6, 2, 2, 2], 2, True),                    # target 10: (4,6) (4,2,2,2)
    ([3, 3, 3, 3], 2, True),                          # (3,3) (3,3)
    ([1] * 16, 4, True),                              # mỗi rổ 4 số 1 (vi phạm tần suất của đề nhưng hợp lệ cho thuật toán)
    (list(range(1, 17)), 8, True),                    # tổng 136, target 17: (1,16) (2,15) ... (8,9)
    ([10000] * 4, 4, True),                           # giá trị lớn nhất
    ([10000, 9999, 1, 10000, 9999, 1], 4, True),      # tổng 40000: (10000) (10000) (9999,1) (9999,1)
    ([10000, 9999, 1, 10000, 9999, 1], 3, False),     # 40000 % 3 != 0
    ([10000, 9999, 1, 10000, 9999, 1], 2, True),      # target 20000: (10000,10000) (9999,1,9999,1)
    ([7, 9, 6, 4, 9, 4, 5, 4, 6, 5, 5], 4, False),    # chia hết (64 / 4 = 16) nhưng không chia được: ca "chia hết mà sai"
]
for nums, k, expected in edge:
    for name, fn in solutions.items():
        if name == "brute" and k ** len(nums) > 50000:
            continue
        assert fn(nums, k) == expected, (name, nums, k, fn(nums, k), expected)
print("12.2 edge case: OK", len(edge), "ca")

# 12.3 phản ví dụ: các cách sai
def wrong_no_div(nums, k):
    """Chỉ tìm k-1 tập có tổng total//k, không kiểm tra tổng % k == 0."""
    target = sum(nums) // k
    nums = sorted(nums, reverse=True)
    n = len(nums)
    used = [False] * n
    def dfs(start, cur, left):
        if left == 1:
            return True
        if cur == target:
            return dfs(0, 0, left - 1)
        for i in range(start, n):
            if used[i] or cur + nums[i] > target:
                continue
            used[i] = True
            if dfs(i + 1, cur + nums[i], left):
                return True
            used[i] = False
        return False
    return dfs(0, 0, k)

def wrong_no_undo(nums, k):
    total = sum(nums)
    if total % k:
        return False
    s = total // k
    nums = sorted(nums, reverse=True)
    cur = [0] * k
    def dfs(i):
        if i == len(nums):
            return True
        for j in range(k):
            if cur[j] + nums[i] > s:
                continue
            cur[j] += nums[i]
            if dfs(i + 1):
                return True
            # QUÊN: cur[j] -= nums[i]
        return False
    return dfs(0)

def wrong_greedy(nums, k):
    total = sum(nums)
    if total % k:
        return False
    s = total // k
    rest = sorted(nums, reverse=True)
    for _ in range(k):
        cur, left = 0, []
        for v in rest:
            if cur + v <= s:
                cur += v
            else:
                left.append(v)
        if cur != s:
            return False
        rest = left
    return not rest

def wrong_dp_sum(nums, k):
    """dp[mask] lưu tổng thay vì phần dư, rồi kiểm dp[full] == target (sai)."""
    total = sum(nums)
    if total % k:
        return False
    target = total // k
    n = len(nums)
    dp = [-1] * (1 << n)
    dp[0] = 0
    for mask in range(1 << n):
        if dp[mask] < 0:
            continue
        for i in range(n):
            if not mask >> i & 1 and dp[mask | 1 << i] < 0 and dp[mask] + nums[i] <= target:
                dp[mask | 1 << i] = dp[mask] + nums[i]
    return dp[(1 << n) - 1] == target

wrongs = [
    ("không kiểm tra chia hết", wrong_no_div, [1, 1, 1, 1, 1], 2),
    ("quên hoàn tác", wrong_no_undo, [6, 6, 4, 3, 3, 4], 2),
    ("tham lam lấy lớn nhất vừa", wrong_greedy, [9, 8, 6, 5, 3, 9], 2),
    ("dp lưu tổng thay vì phần dư", wrong_dp_sum, [1, 2, 3, 4], 2),
]
for name, fn, nums, k in wrongs:
    truth = can_brute(nums, k) if len(nums) <= 7 else can_bitmask(nums, k)
    print(f"{name:28} {nums} k={k}: sai = {fn(nums, k)}, đúng = {truth}")
    assert fn(nums, k) != truth, name
print("12.3 phản ví dụ: OK (các cách sai đều bị lộ)")

12.1 ví dụ đề: OK


12.2 edge case: OK 19 ca
không kiểm tra chia hết      [1, 1, 1, 1, 1] k=2: sai = True, đúng = False
quên hoàn tác                [6, 6, 4, 3, 3, 4] k=2: sai = False, đúng = True
tham lam lấy lớn nhất vừa    [9, 8, 6, 5, 3, 9] k=2: sai = False, đúng = True
dp lưu tổng thay vì phần dư  [1, 2, 3, 4] k=2: sai = False, đúng = True
12.3 phản ví dụ: OK (các cách sai đều bị lộ)


In [25]:
# 12.4 ngẫu nhiên nhỏ so với oracle vét cạn
rng = random.Random(2698)
for _ in range(400):
    n = rng.randint(1, 7)
    k = rng.randint(1, n)
    nums = [rng.randint(1, rng.choice([3, 6, 12])) for _ in range(n)]
    ref = can_bitmask(nums, k) if k ** n > 30000 else can_brute(nums, k)
    for name, fn in solutions.items():
        if name == "brute" and k ** n > 30000:
            continue
        assert fn(nums, k) == ref, (name, nums, k)
print("12.4 ngẫu nhiên nhỏ so với oracle: OK")

# 12.5 ngẫu nhiên n tới 16: các cách nhanh khớp nhau (bitmask DP làm chuẩn); giá trị tới 10^4, tần suất <= 4
rng = random.Random(77)
fast = ["buckets", "subsets", "bitmask", "cover", "final"]
n_true = 0
for _ in range(60):
    n = rng.randint(8, 16)
    k = rng.randint(1, n)
    if rng.random() < 0.6:
        # xây mảng chia được: k nhóm cùng tổng
        target = rng.randint(max(n, 8), 10000 if rng.random() < 0.3 else 60)
        try:
            nums = make_true_instance(rng, n, k, target)
        except ValueError:
            nums = [rng.randint(1, 10) for _ in range(n)]
    else:
        nums = [rng.randint(1, rng.choice([5, 20, 10000])) for _ in range(n)]
    while max(Counter(nums).values()) > 4:
        nums[rng.randrange(n)] = rng.randint(1, 10000)
    res = {name: solutions[name](nums, k) for name in fast}
    assert len(set(res.values())) == 1, (nums, k, res)
    n_true += res["final"]
print(f"12.5 ngẫu nhiên n tới 16: OK ({n_true} đúng / {60 - n_true} sai)")

12.4 ngẫu nhiên nhỏ so với oracle: OK


12.5 ngẫu nhiên n tới 16: OK (20 đúng / 40 sai)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không kiểm tra `tổng % k`

Nếu chỉ tìm `k - 1` tập có tổng `tổng // k` rồi tin phần còn lại "tự khớp", ta nhận nhầm các mảng mà tổng không chia hết.

In [26]:
nums, k = [1, 1, 1, 1, 1], 2
print("sai :", wrong_no_div(nums, k), "| đúng:", Solution().canPartitionKSubsets(nums, k))
assert wrong_no_div(nums, k) is True and Solution().canPartitionKSubsets(nums, k) is False

sai : True | đúng: False


Vì sao sai: tổng 5, `5 // 2 = 2`: tìm được một tập `(1,1)` nên tưởng "đủ", nhưng phần dư `(1,1,1)` có tổng 3 ≠ 2. **Đúng:** `if total % k: return False` trước tiên.

### Lỗi 2 — Quên hoàn tác (backtrack)

Sau khi thử một nhánh thất bại phải trả `cur[j]` về như cũ, nếu không các nhánh sau thấy trạng thái bẩn.

In [27]:
nums, k = [6, 6, 4, 3, 3, 4], 2
print("sai :", wrong_no_undo(nums, k), "| đúng:", Solution().canPartitionKSubsets(nums, k), "(chia: (6,6,... ) tổng 26, target 13: (6,4,3) (6,4,3))")
assert wrong_no_undo(nums, k) is False and Solution().canPartitionKSubsets(nums, k) is True

sai : False | đúng: True (chia: (6,6,... ) tổng 26, target 13: (6,4,3) (6,4,3))


Vì sao sai: sau nhánh thất bại đầu tiên, `cur[j]` còn giữ phần tử cũ nên rổ trông đầy hơn thực tế. **Đúng:** `cur[j] -= nums[i]` ngay sau lời gọi đệ quy thất bại.

### Lỗi 3 — Chọn tham lam "lấy số lớn nhất còn vừa"

Mỗi lần dựng một rổ bằng cách lấy lần lượt số lớn nhất vừa chỗ trông hợp lý nhưng **không** đúng.

In [28]:
nums, k = [4, 3, 2, 3, 5, 2, 1], 4
print("tham lam:", wrong_greedy(nums, k), "| đúng:", Solution().canPartitionKSubsets(nums, k))
assert wrong_greedy(nums, k) is True      # ví dụ này tham lam may mắn đúng

# tìm một phản ví dụ thật sự của tham lam bằng tìm kiếm có seed
rng = random.Random(1)
found = None
for _ in range(20000):
    n = rng.randint(4, 7)
    k = rng.randint(2, 3)
    arr = [rng.randint(1, 9) for _ in range(n)]
    if can_brute(arr, k) and not wrong_greedy(arr, k):
        found = (arr, k)
        break
print("phản ví dụ tìm được:", found, "| đúng =", can_brute(*found), "| tham lam =", wrong_greedy(*found))
assert found is not None and can_brute(*found) is True and wrong_greedy(*found) is False

tham lam: True | đúng: True
phản ví dụ tìm được: ([7, 7, 3, 6, 9, 6, 2], 2) | đúng = True | tham lam = False


Vì sao sai: lấy số lớn nhất vừa chỗ cho rổ đầu có thể "ăn" mất những số mà rổ sau cần để khớp đúng `target`. Bài toán tổng quát là NP-khó nên cần **thử mọi khả năng** (quay lui). **Đúng:** quay lui hoặc bitmask DP.

### Lỗi 4 — `dp[mask]` lưu tổng thay vì phần dư

Nếu `dp[mask]` là tổng thô của `mask` (và chỉ cho đi tiếp khi `<= target`) thì sau khi đầy một rổ (`tổng == target`) ta không thể thêm số vào rổ mới (`target + x > target`).

In [29]:
nums, k = [1, 2, 3, 4], 2
print("sai :", wrong_dp_sum(nums, k), "| đúng:", Solution().canPartitionKSubsets(nums, k))
assert wrong_dp_sum(nums, k) is False and Solution().canPartitionKSubsets(nums, k) is True

sai : False | đúng: True


Vì sao sai: với `k = 2` và `target = 5`, ghép `(1, 4)` xong thì dp = 5, thêm `2` cho `7 > 5` bị chặn nên không bao giờ tới `dp[full]`. **Đúng:** lưu `(dp + x) % target`, để rổ đầy thì phần dư về 0 và bắt đầu rổ mới.

### Lỗi 5 — Không cắt tỉa / không sắp xếp: chạy quá chậm

Không sắp giảm dần và không bỏ rổ trùng tổng, quay lui sinh ra số nhánh khổng lồ trên các mảng "sai" (phải thử hết rồi mới kết luận False).

In [30]:
def count_calls(nums, k, sort_desc=True, skip_same=True):
    total = sum(nums)
    if total % k:
        return 0
    s = total // k
    nums = sorted(nums, reverse=sort_desc)
    cur = [0] * k
    cnt = 0
    def dfs(i):
        nonlocal cnt
        cnt += 1
        if i == len(nums):
            return True
        for j in range(k):
            if skip_same and j and cur[j] == cur[j - 1]:
                continue
            if cur[j] + nums[i] > s:
                continue
            cur[j] += nums[i]
            if dfs(i + 1):
                return True
            cur[j] -= nums[i]
        return False
    dfs(0)
    return cnt

arr, k = [7, 9, 6, 4, 9, 4, 5, 4, 6, 5, 5], 4          # tổng 64, target 16, nhưng KHÔNG chia được (phải thử hết mới biết)
print("tổng", sum(arr), "target", sum(arr) // k)
for sd, ss in [(True, True), (False, True), (True, False), (False, False)]:
    print(f"sắp giảm dần={sd!s:5} bỏ rổ trùng={ss!s:5}: số lần gọi dfs = {count_calls(arr, k, sd, ss):>9,}")

tổng 64 target 16
sắp giảm dần=True  bỏ rổ trùng=True : số lần gọi dfs =       374
sắp giảm dần=False bỏ rổ trùng=True : số lần gọi dfs =     2,074
sắp giảm dần=True  bỏ rổ trùng=False: số lần gọi dfs =    20,441
sắp giảm dần=False bỏ rổ trùng=False: số lần gọi dfs =   144,673


Vì sao chậm: các nhánh đối xứng (hoán đổi nhãn rổ) và số nhỏ đặt trước làm cây tìm kiếm phình to. **Đúng:** sắp giảm dần và `if j and cur[j] == cur[j - 1]: continue`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu `n <= 16` / `n <= 20`** + "chia nhóm"/"chọn tập con" -> bitmask hoặc quay lui có cắt tỉa.
- **Chia thành `k` phần có cùng tổng** -> trước hết `tổng % k`, rồi `target = tổng / k`; phần tử `> target` là loại ngay.
- **Đối xứng giữa các rổ giống hệt** -> bỏ nhánh (rổ cùng tổng hiện tại).
- **Phần tử lớn trước** để thất bại sớm.
- **Bitmask DP "phần dư"**: tổng của `mask` quyết định rổ hiện tại là `tổng % target`; dùng `dp[mask]` làm cờ khả thi + phần dư.
- Bit thấp nhất `mask & -mask` giúp liệt kê phân hoạch **không lặp** (chỉ xét tập chứa phần tử nhỏ nhất còn lại).
- Biến thể hay gặp: Partition Equal Subset Sum (416, `k = 2` cho phép DP knapsack), Matchsticks to Square (473, `k = 4` hình vuông), Fair Distribution of Cookies (2305).
- Nhớ **hoàn tác** mọi biến đã thay đổi trong quay lui.

## 15. Biến thể

1. **Partition Equal Subset Sum (416):** `k = 2`. Có lời giải knapsack `O(n * tổng)`, nhanh hơn nhiều khi `tổng` nhỏ.
2. **Matchsticks to Square (473):** `k = 4`, mỗi que dùng đúng một lần -> cùng bài với `k = 4`.
3. **Trả về một cách chia cụ thể** (bài tập 5).
4. **Đếm số cách chia** thành `k` tập cùng tổng (không phân biệt thứ tự các tập): đếm bằng bao phủ chính xác với bit thấp nhất.
5. **Fair Distribution of Cookies (2305):** chia `n` gói bánh cho `k` bạn để **tối thiểu hóa tổng lớn nhất**: chặt nhị phân theo giới hạn hoặc quay lui.

Dưới đây là code cho biến thể 1 (knapsack) và 4 (đếm), kiểm bằng oracle.

In [31]:
def can_partition_equal(nums: List[int]) -> bool:
    total = sum(nums)
    if total % 2:
        return False
    half = total // 2
    reach = 1                                   # bitset: bit t = 1 nếu đạt được tổng t
    for v in nums:
        reach |= reach << v
    return reach >> half & 1 == 1

def count_partitions(nums: List[int], k: int) -> int:
    """Số cách chia thành k tập cùng tổng (tập không có thứ tự)."""
    total = sum(nums)
    if total % k:
        return 0
    target = total // k
    n = len(nums)
    s = [0] * (1 << n)
    by_low = {}
    for mask in range(1, 1 << n):
        low = mask & -mask
        s[mask] = s[mask ^ low] + nums[low.bit_length() - 1]
        if s[mask] == target:
            by_low.setdefault(low, []).append(mask)

    @lru_cache(None)
    def ways(rem):
        if rem == 0:
            return 1
        low = rem & -rem
        return sum(ways(rem ^ g) for g in by_low.get(low, []) if g & rem == g)

    return ways((1 << n) - 1)

def count_oracle(nums, k):
    n = len(nums)
    seen = set()
    for assign in product(range(k), repeat=n):
        sums = [0] * k
        groups = [[] for _ in range(k)]
        for i, b in enumerate(assign):
            sums[b] += nums[i]
            groups[b].append(i)
        if len(set(sums)) == 1:
            seen.add(frozenset(frozenset(g) for g in groups))
    return len(seen)

assert can_partition_equal([1, 5, 11, 5]) is True and can_partition_equal([1, 2, 3, 5]) is False
for arr in all_arrays(5, 4):
    assert can_partition_equal(list(arr)) == can_brute(list(arr), 2), arr
print("số cách chia [1,2,3,4] thành 2 tập cùng tổng:", count_partitions([1, 2, 3, 4], 2))
for _ in range(150):
    n = random.Random(_).randint(1, 6)
    arr = [random.Random(_ * 7 + i).randint(1, 4) for i in range(n)]
    k = random.Random(_ * 3).randint(1, n)
    assert count_partitions(arr, k) == count_oracle(arr, k), (arr, k)
print("Biến thể: OK")

số cách chia [1,2,3,4] thành 2 tập cùng tổng: 1


Biến thể: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Gán mọi số vào rổ | Định nghĩa trực tiếp | `O(k^n n)` | `O(n)` | Làm oracle, `n` rất nhỏ |
| 2. Quay lui theo rổ | Cắt tỉa: giảm dần, vượt, rổ trùng | xấu `O(k^n)`, thực tế nhỏ | `O(n + k)` | **Cách nộp chính**, ngắn gọn |
| 3. Dựng từng tập con | Tổ hợp theo chỉ số, đóng rổ khi đủ | mũ nhưng nhỏ | `O(n)` | Khi muốn suy nghĩ "một rổ một lần" |
| 4. Bitmask DP | `dp[mask]` phần dư | `O(n 2^n)` | `O(2^n)` | Thời gian **ổn định**, không phụ thuộc cắt tỉa |
| 5. Bao phủ chính xác | Tập tốt + bit thấp nhất | `O(2^n)`-cỡ | `O(2^n)` | Khi cần đếm/liệt kê phân hoạch |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
print(sum([4, 3, 2, 3, 5, 2, 1]) % 4)
print(sorted([4, 3, 2, 3, 5, 2, 1], reverse=True))
print(bin((1 << 3) | (1 << 0)), (0b1010 & -0b1010))
```

In [32]:
# Bài 1: ghi dự đoán của bạn (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `nums = [1, 2, 3, 4]` và `k = 2`: ghi `target`, một cách chia (list các list) và với `k = 3` kết luận True/False cùng lý do (một câu trong comment).

In [33]:
# Bài 2: điền kết quả tính tay
hand_target = None
hand_groups = None        # ví dụ [[...], [...]]
hand_k3 = None            # True/False

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành quay lui theo rổ.

In [34]:
def exercise_3_buckets(nums: List[int], k: int) -> bool:
    total = sum(nums)
    # TODO 1: nếu total không chia hết k thì trả False; đặt target = total // k
    # TODO 2: sắp nums giảm dần; nếu phần tử lớn nhất > target thì trả False
    cur = [0] * k

    def dfs(i: int) -> bool:
        # TODO 3: nếu i == len(nums) thì trả True
        for j in range(k):
            # TODO 4: bỏ qua rổ j nếu trùng tổng với rổ j - 1 hoặc nếu cur[j] + nums[i] > target
            # TODO 5: đặt nums[i] vào rổ j, đệ quy, và HOÀN TÁC nếu thất bại
            pass
        return False

    return None


# assert exercise_3_buckets([4, 3, 2, 3, 5, 2, 1], 4) is True
# assert exercise_3_buckets([1, 2, 3, 4], 3) is False
# assert exercise_3_buckets([1, 2, 3, 4], 2) is True

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, hãy tự viết `exercise_4_bitmask(nums, k)` bằng bitmask DP với `dp[mask]` là phần dư của rổ hiện tại.

In [35]:
def exercise_4_bitmask(nums: List[int], k: int) -> bool:
    # TODO
    pass


# assert exercise_4_bitmask([4, 3, 2, 3, 5, 2, 1], 4) is True
# assert exercise_4_bitmask([1, 2, 3, 4], 3) is False
# assert exercise_4_bitmask([1, 2, 3, 4], 2) is True

### Bài 5 — Mini challenge: trả về một cách chia (khó)

Viết `exercise_5_partition(nums, k)` trả về **list `k` list** (các giá trị của mỗi tập) nếu chia được, ngược lại `None`. Có thể có nhiều đáp án đúng, kiểm tra bằng hàm hợp lệ: mỗi giá trị của `nums` dùng đúng số lần, `k` tập cùng tổng.

```python
exercise_5_partition([4, 3, 2, 3, 5, 2, 1], 4)  -> ví dụ [[5], [4, 1], [3, 2], [3, 2]] (thứ tự bất kỳ)
exercise_5_partition([1, 2, 3, 4], 3)           -> None
```
Gợi ý: Cách 2, thêm `groups[j].append(...)` / `.pop()` song song với `cur`.

In [36]:
def exercise_5_partition(nums: List[int], k: int) -> Optional[List[List[int]]]:
    # TODO
    pass


def valid_partition(nums: List[int], k: int, groups) -> bool:
    if groups is None or len(groups) != k or any(len(g) == 0 for g in groups):
        return False
    if sorted(x for g in groups for x in g) != sorted(nums):
        return False
    return len({sum(g) for g in groups}) == 1


# for nums, k, expected in [([4, 3, 2, 3, 5, 2, 1], 4, True), ([1, 2, 3, 4], 3, False), ([1, 2, 3, 4], 2, True)]:
#     res = exercise_5_partition(nums, k)
#     assert (res is not None) == expected
#     if expected:
#         assert valid_partition(nums, k, res)

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: tổng của danh sách đã thấy ở ví dụ 1.
- Mức 2: tổng là 20; `20 % 4` bằng 0. `bin(...)` in kèm tiền tố `0b`.
- Mức 3: `0`, `[5, 4, 3, 3, 2, 2, 1]`, `0b1001 2` (`0b1010 & -0b1010` là bit thấp nhất = 2).

**Bài 2**
- Mức 1: tổng 10, `k = 2` nên `target = 5`.
- Mức 2: tìm cặp có tổng 5: `(1, 4)` và `(2, 3)`.
- Mức 3: `hand_target = 5`, `[[1, 4], [2, 3]]`; với `k = 3`: `10 % 3 != 0` nên `False`.

**Bài 3**
- Mức 1: hai điều kiện loại sớm: `total % k` và `max > target`.
- Mức 2: trong vòng `for j`: `if j and cur[j] == cur[j - 1]: continue` và `if cur[j] + nums[i] > target: continue`.
- Mức 3: `cur[j] += nums[i]; if dfs(i + 1): return True; cur[j] -= nums[i]`; cuối `return dfs(0)`.

**Bài 4**
- Mức 1: `dp = [-1] * (1 << n)`, `dp[0] = 0`.
- Mức 2: với `mask` có `dp[mask] >= 0` và `i` chưa có: nếu `dp[mask] + nums[i] <= target` thì `dp[mask | 1 << i] = (dp[mask] + nums[i]) % target`.
- Mức 3: `return dp[(1 << n) - 1] == 0` (nhớ kiểm tra chia hết và `max <= target` trước).

**Bài 5**
- Mức 1: dùng Cách 2, thêm `groups = [[] for _ in range(k)]`.
- Mức 2: khi đặt `nums[i]` vào rổ `j`: `groups[j].append(nums[i])`; khi hoàn tác: `groups[j].pop()`.
- Mức 3: khi `dfs(len(nums))` thành công, trả `[list(g) for g in groups]` (sao chép!).

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [37]:
print(sum([4, 3, 2, 3, 5, 2, 1]) % 4)
print(sorted([4, 3, 2, 3, 5, 2, 1], reverse=True))
print(bin((1 << 3) | (1 << 0)), (0b1010 & -0b1010))

assert sum([4, 3, 2, 3, 5, 2, 1]) % 4 == 0
assert sorted([4, 3, 2, 3, 5, 2, 1], reverse=True) == [5, 4, 3, 3, 2, 2, 1]
assert bin((1 << 3) | (1 << 0)) == "0b1001" and (0b1010 & -0b1010) == 2

0
[5, 4, 3, 3, 2, 2, 1]
0b1001 2


### Đáp án Bài 2

`target = 5`; chia `[[1, 4], [2, 3]]`; với `k = 3`: tổng 10 không chia hết cho 3 nên `False`.

In [38]:
hand_target_solution = 5
hand_groups_solution = [[1, 4], [2, 3]]
hand_k3_solution = False
assert sum([1, 2, 3, 4]) // 2 == hand_target_solution
assert all(sum(g) == hand_target_solution for g in hand_groups_solution)
assert sorted(sum(hand_groups_solution, [])) == [1, 2, 3, 4]
assert Solution().canPartitionKSubsets([1, 2, 3, 4], 3) is hand_k3_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [39]:
def exercise_3_buckets_solution(nums: List[int], k: int) -> bool:
    total = sum(nums)
    if total % k:
        return False
    target = total // k
    nums = sorted(nums, reverse=True)
    if nums[0] > target:
        return False
    cur = [0] * k

    def dfs(i: int) -> bool:
        if i == len(nums):
            return True
        for j in range(k):
            if j and cur[j] == cur[j - 1]:
                continue
            if cur[j] + nums[i] > target:
                continue
            cur[j] += nums[i]
            if dfs(i + 1):
                return True
            cur[j] -= nums[i]
        return False

    return dfs(0)

assert exercise_3_buckets_solution([4, 3, 2, 3, 5, 2, 1], 4) is True
assert exercise_3_buckets_solution([1, 2, 3, 4], 3) is False
assert exercise_3_buckets_solution([1, 2, 3, 4], 2) is True
rng = random.Random(3)
for _ in range(200):
    n = rng.randint(1, 7)
    k = rng.randint(1, n)
    nums = [rng.randint(1, 8) for _ in range(n)]
    assert exercise_3_buckets_solution(nums, k) == can_brute(nums, k), (nums, k)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [40]:
def exercise_4_bitmask_solution(nums: List[int], k: int) -> bool:
    total = sum(nums)
    if total % k:
        return False
    target = total // k
    if max(nums) > target:
        return False
    n = len(nums)
    dp = [-1] * (1 << n)
    dp[0] = 0
    for mask in range(1 << n):
        if dp[mask] < 0:
            continue
        for i in range(n):
            if mask >> i & 1:
                continue
            if dp[mask] + nums[i] <= target:
                nm = mask | (1 << i)
                if dp[nm] < 0:
                    dp[nm] = (dp[mask] + nums[i]) % target
    return dp[(1 << n) - 1] == 0

assert exercise_4_bitmask_solution([4, 3, 2, 3, 5, 2, 1], 4) is True
assert exercise_4_bitmask_solution([1, 2, 3, 4], 3) is False
assert exercise_4_bitmask_solution([1, 2, 3, 4], 2) is True
rng = random.Random(4)
for _ in range(200):
    n = rng.randint(1, 7)
    k = rng.randint(1, n)
    nums = [rng.randint(1, 8) for _ in range(n)]
    assert exercise_4_bitmask_solution(nums, k) == can_brute(nums, k), (nums, k)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [41]:
def exercise_5_partition_solution(nums: List[int], k: int) -> Optional[List[List[int]]]:
    total = sum(nums)
    if total % k:
        return None
    target = total // k
    nums = sorted(nums, reverse=True)
    if nums[0] > target:
        return None
    cur = [0] * k
    groups = [[] for _ in range(k)]

    def dfs(i: int) -> bool:
        if i == len(nums):
            return True
        for j in range(k):
            if j and cur[j] == cur[j - 1]:
                continue
            if cur[j] + nums[i] > target:
                continue
            cur[j] += nums[i]
            groups[j].append(nums[i])
            if dfs(i + 1):
                return True
            groups[j].pop()
            cur[j] -= nums[i]
        return False

    return [list(g) for g in groups] if dfs(0) else None

for nums, k, expected in [([4, 3, 2, 3, 5, 2, 1], 4, True), ([1, 2, 3, 4], 3, False), ([1, 2, 3, 4], 2, True)]:
    res = exercise_5_partition_solution(nums, k)
    assert (res is not None) == expected
    if expected:
        assert valid_partition(nums, k, res), res
rng = random.Random(5)
for _ in range(200):
    n = rng.randint(1, 8)
    k = rng.randint(1, n)
    nums = [rng.randint(1, 8) for _ in range(n)]
    res = exercise_5_partition_solution(nums, k)
    assert (res is not None) == can_bitmask(nums, k), (nums, k)
    if res is not None:
        assert valid_partition(nums, k, res), (nums, k, res)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (quay lui theo rổ có cắt tỉa)

```python
class Solution:
    def canPartitionKSubsets(self, nums: List[int], k: int) -> bool:
        total = sum(nums)
        if total % k:                                  # điều kiện cần 1
            return False
        s = total // k
        nums = sorted(nums, reverse=True)              # số lớn trước
        if nums[0] > s:                                # điều kiện cần 2
            return False
        cur = [0] * k                                  # tổng hiện tại của mỗi rổ

        def dfs(i: int) -> bool:
            if i == len(nums):
                return True
            for j in range(k):
                if j and cur[j] == cur[j - 1]:         # rổ đối xứng -> bỏ
                    continue
                if cur[j] + nums[i] > s:               # vượt mục tiêu -> bỏ
                    continue
                cur[j] += nums[i]
                if dfs(i + 1):
                    return True
                cur[j] -= nums[i]                      # HOÀN TÁC
            return False

        return dfs(0)
```

Công thức nhớ nhanh: `target = tổng / k`; bitmask DP: `dp[mask | 1<<i] = (dp[mask] + nums[i]) % target` nếu `dp[mask] + nums[i] <= target`; đáp án `dp[full] == 0`.

### Checklist trước khi submit

- [ ] Đã kiểm tra `tổng % k == 0` và `max(nums) <= target` trước tiên?
- [ ] Sắp giảm dần và bỏ rổ có cùng tổng với rổ trước?
- [ ] Mọi thay đổi (`cur[j] += ...`, `used[i] = True`) đều được **hoàn tác** khi nhánh thất bại?
- [ ] Nếu dùng bitmask: `dp` kích thước `1 << n`, `dp[0] = 0`, lưu **phần dư** (`% target`)?
- [ ] Đã thử `k = 1`, `k = n` (mọi phần tử bằng nhau / khác nhau), tổng không chia hết, phần tử lớn hơn `target`?
- [ ] Đã thử mảng 16 phần tử (đo thời gian)?
- [ ] Tên method đúng `canPartitionKSubsets(self, nums, k)` và trả `bool`?